In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cell 1 ▶ Install required packages
!pip install timm torchvision tifffile imagecodecs --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 MB 13.1 MB/s eta 0:00:00


In [ ]:
# Cell 2 ▶ Consolidated imports
import os
import math
import time

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

import timm
from torchvision import transforms, models

from tifffile import imread

In [ ]:
# Cell 1 ▶ Dataset (224×224 targets, no down-scaling)
class LSTDataset(Dataset):
    def __init__(self, df, patches_dir, weather_cols):
        self.df           = df.reset_index(drop=True)
        self.patches_dir  = patches_dir
        self.weather_cols = weather_cols
        self.transform    = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),          # resize image
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.loc[idx]
        arr  = imread(os.path.join(self.patches_dir, row["filename"])
                     ).astype(np.float32)            # (4,H,W)

        # --- inputs -------------------------------------------------
        img_np = arr[[1,2,3]].transpose(1,2,0).astype(np.uint8)
        img    = self.transform(img_np)              # [3,224,224]

        # --- target (LST) at *full* 224×224 --------------------------
        lst    = arr[0]                              # (H,W)
        lst    = torch.tensor(lst, dtype=torch.float32).unsqueeze(0)
        lst    = F.interpolate(lst.unsqueeze(0), size=(224,224),
                               mode='bilinear', align_corners=False
                              ).squeeze(0)           # [1,224,224]

        # --- meteorology vector -------------------------------------
        weather = torch.tensor(
            row[self.weather_cols].values.astype(np.float32)
        )
        return img, weather, lst


In [ ]:
# define which meteorological columns to pull
weather_cols = [
    "air_temp_C",
    "dew_point_C",
    "relative_humidity_percent",
    "wind_speed_m_s",
    "precipitation_in",
]

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/PatchedOutput/patch_with_meteo.csv")
for col in weather_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=weather_cols + ["filename"]).reset_index(drop=True)

patches_dir = "/content/drive/MyDrive/PatchedOutput_Cleaned"
dataset     = LSTDataset(df, patches_dir, weather_cols)
train_sz    = int(0.8 * len(dataset))
val_sz      = len(dataset) - train_sz
train_ds, val_ds = random_split(dataset, [train_sz, val_sz])

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=0, pin_memory=False)

In [ ]:
# Cell 4 ▶ ViT + weather → 224×224 decoder
import math, torch.nn as nn, timm

class PretrainedViTLSTModel(nn.Module):
    def __init__(self,
                 weather_dim=5,
                 hidden_dim=768,
                 vit_name="vit_base_patch16_224",
                 num_layers=2,
                 num_heads=8):
        super().__init__()

        # ---- ViT backbone -----------------------------------------
        self.vit = timm.create_model(vit_name, pretrained=True, num_classes=0)
        for p in self.vit.parameters():
            p.requires_grad = False          # freeze by default

        # ---- project weather vector -> token ----------------------
        self.weather_proj = nn.Linear(weather_dim, hidden_dim)

        # ---- tiny transformer for fusion --------------------------
        enc = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim*4,
            dropout=0.1)
        self.transformer = nn.TransformerEncoder(enc, num_layers)

        # ---- decoder: 14→28→56→112→224 ----------------------------
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(hidden_dim, hidden_dim//2, 2, 2),  # 14→28
            nn.BatchNorm2d(hidden_dim//2), nn.ReLU(inplace=True),

            nn.ConvTranspose2d(hidden_dim//2, hidden_dim//4, 2, 2),  # 28→56
            nn.BatchNorm2d(hidden_dim//4), nn.ReLU(inplace=True),

            nn.ConvTranspose2d(hidden_dim//4, hidden_dim//8, 2, 2),  # 56→112
            nn.BatchNorm2d(hidden_dim//8), nn.ReLU(inplace=True),

            nn.ConvTranspose2d(hidden_dim//8, 1, 2, 2)               # 112→224
        )

    def forward(self, images, weather):
        feats   = self.vit.forward_features(images)       # [B,197,768]
        cls_tok = feats[:, :1]                            # [B,1,768]
        patch_t = feats[:, 1:]                            # [B,196,768]

        w_tok   = self.weather_proj(weather).unsqueeze(1) # [B,1,768]
        tokens  = torch.cat([patch_t, w_tok, cls_tok], 1) # [B,198,768]

        t = self.transformer(tokens.permute(1,0,2)).permute(1,0,2)
        patch_out = t[:, :-2, :]                          # drop weather+CLS

        B, N, D = patch_out.shape        # N = 196
        G       = int(math.sqrt(N))      # 14
        x       = patch_out.transpose(1,2).view(B, D, G, G)  # [B,768,14,14]
        return self.deconv(x)            # [B,1,224,224]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = PretrainedViTLSTModel(
    weather_dim=len(weather_cols),
    hidden_dim=768,
    vit_name="vit_base_patch16_224",
    num_layers=2,
    num_heads=8
).to(device)

# unfreeze last ViT blocks:
for name, param in model.vit.named_parameters():
    if any(layer in name for layer in ["blocks.10", "blocks.11", "norm"]):
        param.requires_grad = True

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [ ]:
opt       = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-2
)
loss_fn   = nn.SmoothL1Loss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt, mode='min', factor=0.5, patience=3, verbose=True
)

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [ ]:
# Cell 0 ▶ install LZW support
!pip install imagecodecs --quiet


In [ ]:
from tifffile import imread


In [ ]:
from tqdm import tqdm
import os
from pathlib import Path
import torch

# Create a directory on your Drive for checkpoints
save_dir = Path("/content/drive/MyDrive/Model_vit_mlp_transformer_30m_Checkpoints")
save_dir.mkdir(parents=True, exist_ok=True)

num_epochs = 10
start_ep   = 0  # or pick up from a checkpoint

for epoch in range(start_ep, num_epochs):
    # — Train —
    model.train()
    train_loss   = 0.0
    seen_samples = 0
    train_bar    = tqdm(train_loader, desc=f"Epoch {epoch+1:02d} Train")
    for imgs, weather, tgt in train_bar:
        imgs, weather, tgt = imgs.to(device), weather.to(device), tgt.to(device)

        opt.zero_grad()
        out  = model(imgs, weather)
        loss = loss_fn(out, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        batch_val     = loss.item()
        n             = imgs.size(0)
        train_loss   += batch_val * n
        seen_samples += n
        avg_train     = train_loss / seen_samples

        train_bar.set_postfix(
            batch_loss=f"{batch_val:.4f}",
            avg_loss  =f"{avg_train:.4f}"
        )

    train_rmse = (train_loss / len(train_loader.dataset))**0.5

    # — Validate —
    model.eval()
    val_loss = 0.0
    seen_val = 0
    val_bar  = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}   Val ")
    with torch.no_grad():
        for imgs, weather, tgt in val_bar:
            imgs, weather, tgt = imgs.to(device), weather.to(device), tgt.to(device)
            out       = model(imgs, weather)
            batch_val = loss_fn(out, tgt).item()
            n         = imgs.size(0)
            val_loss += batch_val * n
            seen_val += n
            avg_val   = val_loss / seen_val

            val_bar.set_postfix(
                batch_loss=f"{batch_val:.4f}",
                avg_loss  =f"{avg_val:.4f}"
            )

    val_rmse = (val_loss / len(val_loader.dataset))**0.5

    # Step the scheduler
    scheduler.step(val_loss)

    # Print metrics
    print(f"Epoch {epoch+1:02d} ▶ Train RMSE: {train_rmse:.3f} | Val RMSE: {val_rmse:.3f}")

    # — Save checkpoint —
    ckpt_path = save_dir / f"cnn_mlp_epoch{epoch+1:02d}.pt"
    torch.save({
        'epoch': epoch+1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'train_rmse': train_rmse,
        'val_rmse': val_rmse
    }, ckpt_path)
    print(f"✅ Saved checkpoint: {ckpt_path}")

print("✅ Training finished")


Epoch 01   Val : 100%|██████████| 715/715 [38:01<00:00,  3.19s/it, avg_loss=0.6137, batch_loss=0.5388]


Epoch 01 ▶ Train RMSE: 1.193 | Val RMSE: 0.783
✅ Saved checkpoint: /content/drive/MyDrive/Model_vit_mlp_transformer_30m_Checkpoints/cnn_mlp_epoch01.pt


Epoch 02   Val : 100%|██████████| 715/715 [00:57<00:00, 12.35it/s, avg_loss=0.2470, batch_loss=0.1237]


Epoch 02 ▶ Train RMSE: 0.742 | Val RMSE: 0.497
✅ Saved checkpoint: /content/drive/MyDrive/Model_vit_mlp_transformer_30m_Checkpoints/cnn_mlp_epoch02.pt


Epoch 03   Val : 100%|██████████| 715/715 [00:57<00:00, 12.37it/s, avg_loss=0.2417, batch_loss=0.1203]


...

Epoch 08 ▶ Train RMSE: 0.427 | Val RMSE: 0.370
✅ Saved checkpoint: /content/drive/MyDrive/Model_vit_mlp_transformer_30m_Checkpoints/cnn_mlp_epoch08.pt


Epoch 09   Val : 100%|██████████| 715/715 [00:57<00:00, 12.42it/s, avg_loss=0.1336, batch_loss=0.0347]


Epoch 09 ▶ Train RMSE: 0.373 | Val RMSE: 0.366
✅ Saved checkpoint: /content/drive/MyDrive/Model_vit_mlp_transformer_30m_Checkpoints/cnn_mlp_epoch09.pt


Epoch 10   Val : 100%|██████████| 715/715 [00:57<00:00, 12.43it/s, avg_loss=0.1252, batch_loss=0.0428]


Epoch 10 ▶ Train RMSE: 0.349 | Val RMSE: 0.354
✅ Saved checkpoint: /content/drive/MyDrive/Model_vit_mlp_transformer_30m_Checkpoints/cnn_mlp_epoch10.pt
✅ Training finished
